# Experimento A — Baseline: DCGAN preentrenado (Volz et al.)

Este notebook analiza el comportamiento del generador DCGAN original de Volz et al. (2018)  
entrenado sobre los **173 fragmentos** del dataset original de Mario Bros.

**Objetivo:** Establecer el punto de partida (*baseline*) que servirá de referencia para  
los experimentos posteriores de expansión de datos (Exp B) y optimización con CMA-ES.

**Modelo:** `netG_epoch_5000.pth` — DCGAN-G con `nz=32`, `nc=10`, entrenado 5000 épocas.  
**Evaluación:** 100 niveles generados aleatoriamente, métricas estructurales promediadas.

## 1. Setup — Imports y configuración de paths

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import torch

# ── Paths relativos desde notebooks/ ──────────────────────────────────────────
NOTEBOOK_DIR   = os.path.abspath('')          # .../NeuralPlumber/notebooks
PROJECT_ROOT   = os.path.abspath('../../')    # .../ML/Proyecto
DAGSTUHL_PATH  = os.path.join(PROJECT_ROOT, 'clone', 'DagstuhlGAN', 'pytorch')
SRC_PATH       = os.path.join(PROJECT_ROOT, 'NeuralPlumber', 'src')
BASELINE_DIR   = os.path.join(PROJECT_ROOT, 'NeuralPlumber', 'experiments', 'baseline')

for p in [DAGSTUHL_PATH, SRC_PATH]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('DAGSTUHL_PATH existe:', os.path.isdir(DAGSTUHL_PATH))
print('SRC_PATH existe     :', os.path.isdir(SRC_PATH))
print('BASELINE_DIR existe :', os.path.isdir(BASELINE_DIR))

## 2. Métricas del experimento A

In [ ]:
# Cargar métricas guardadas
metrics_path = os.path.join(BASELINE_DIR, 'metrics_baseline.json')
with open(metrics_path, 'r') as f:
    data = json.load(f)

n_samples = data['n_samples']
metrics   = data['metrics']

print(f'Muestras evaluadas: {n_samples}\n')

# ── Tabla de métricas ─────────────────────────────────────────────────────────
METRIC_LABELS = {
    'pipe_completeness' : 'Completitud de tuberías',
    'ground_continuity' : 'Continuidad de suelo',
    'gap_traversability': 'Traversabilidad de gaps',
    'enemy_placement'   : 'Colocación de enemigos',
    'structural_avg'    : 'Promedio estructural',
}

GOOD_THRESHOLD = 0.70

print(f"{'Métrica':<35} {'Valor':>8}  {'Estado'}")
print('-' * 65)
for key, label in METRIC_LABELS.items():
    val    = metrics[key]
    estado = 'OK' if val >= GOOD_THRESHOLD else 'BAJO'
    marker = '<<' if val < GOOD_THRESHOLD else '  '
    print(f"  {label:<33} {val:>8.4f}  {estado} {marker}")
print('-' * 65)

# ── Gráfico de barras ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
keys   = list(METRIC_LABELS.values())
values = [metrics[k] for k in METRIC_LABELS]
colors = ['#e74c3c' if v < GOOD_THRESHOLD else '#2ecc71' for v in values]

bars = ax.barh(keys, values, color=colors, edgecolor='#333', linewidth=0.7)
ax.axvline(GOOD_THRESHOLD, color='#555', linestyle='--', linewidth=1.2, label=f'Umbral {GOOD_THRESHOLD}')
ax.set_xlim(0, 1.05)
ax.set_xlabel('Score (0–1)')
ax.set_title(f'Exp A — Métricas baseline  (n={n_samples})', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')

for bar, val in zip(bars, values):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 3. Visualizar niveles de muestra (sample_levels.png)

In [ ]:
img_path = os.path.join(BASELINE_DIR, 'sample_levels.png')

if os.path.exists(img_path):
    img = mpimg.imread(img_path)
    h, w = img.shape[:2]
    fig_w = min(16, w / 80)
    fig_h = fig_w * h / w
    fig, ax = plt.subplots(figsize=(max(fig_w, 14), max(fig_h, 5)))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title('Niveles de muestra — Exp A (baseline)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print(f'Imagen no encontrada en: {img_path}')

## 4. Generación en vivo de un nivel nuevo

Cargamos el modelo DCGAN preentrenado (`netG_epoch_5000.pth`) y generamos un nivel  
nuevo con un vector latente aleatorio para ver el comportamiento del generador en tiempo real.

In [ ]:
from models.dcgan import DCGAN_G, load_compatible
from visualization.level_renderer import render_level

MODEL_PATH = os.path.join(DAGSTUHL_PATH, 'netG_epoch_5000.pth')

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f'Modelo no encontrado: {MODEL_PATH}')

# ── Cargar generador ──────────────────────────────────────────────────────────
NZ, NC, NGF = 32, 10, 64
netG = DCGAN_G(32, NZ, NC, NGF, ngpu=0, n_extra_layers=0)
load_compatible(netG, MODEL_PATH)
netG.eval()
print(f'Modelo cargado desde: {MODEL_PATH}')
total_params = sum(p.numel() for p in netG.parameters())
print(f'Parámetros totales  : {total_params:,}')

In [ ]:
# ── Generar 1 nivel nuevo ─────────────────────────────────────────────────────
torch.manual_seed(42)  # semilla fija para reproducibilidad
z = torch.randn(1, NZ, 1, 1)

with torch.no_grad():
    output = netG(z)          # (1, 10, 32, 32)

level = output[0, :, :14, :28].argmax(dim=0).numpy()  # (14, 28)

unique_tiles, counts = np.unique(level, return_counts=True)
total_tiles = level.size

TILE_NAMES = {
    0: 'Suelo', 1: 'Breakable', 2: 'Vacío', 3: 'Q-lleno',
    4: 'Q-vacío', 5: 'Enemigo', 6: 'Pipe TL', 7: 'Pipe TR',
    8: 'Pipe L', 9: 'Pipe R',
}

print(f'Nivel generado — forma: {level.shape}')
print(f'\nDistribución de tiles ({total_tiles} total):')
for tile_id, count in zip(unique_tiles, counts):
    name = TILE_NAMES.get(tile_id, f'Tile {tile_id}')
    pct  = 100.0 * count / total_tiles
    print(f'  [{tile_id}] {name:<12}: {count:4d} ({pct:5.1f}%)')

# ── Renderizar ────────────────────────────────────────────────────────────────
render_level(level, title='Nivel generado en vivo — Exp A (z aleatorio, seed=42)')

In [ ]:
# ── Comparar 4 niveles con distintos vectores latentes ────────────────────────
from visualization.level_renderer import render_comparison

seeds  = [0, 7, 13, 99]
levels = []
for s in seeds:
    torch.manual_seed(s)
    z = torch.randn(1, NZ, 1, 1)
    with torch.no_grad():
        out = netG(z)
    lv = out[0, :, :14, :28].argmax(dim=0).numpy()
    levels.append(lv)

titles = [f'Nivel con seed={s}' for s in seeds]
render_comparison(levels, titles=titles)

## 5. Análisis de resultados

### Métricas destacadas

| Métrica | Score | Interpretación |
|---|---|---|
| `pipe_completeness` | **0.2075** | Muy bajo: las tuberías rara vez son estructuralmente completas |
| `ground_continuity` | **0.5093** | Medio-bajo: el suelo se fragmenta con frecuencia |
| `gap_traversability`| **0.9100** | Alto: los huecos tienden a ser saltables |
| `enemy_placement`   | **0.9667** | Muy alto: los enemigos aparecen sobre suelo sólido |
| `structural_avg`    | **0.6484** | Promedio: el nivel global queda por debajo del umbral 0.70 |

### ¿Por qué son bajas `pipe_completeness` y `ground_continuity`?

1. **Escasez de datos estructurados**: el dataset original tiene solo **173 fragmentos**.  
   Las tuberías y el suelo continuo son patrones complejos que requieren consistencia  
   espacial a lo largo de múltiples tiles adyacentes. Con tan pocas muestras, el generador  
   no aprendió a reproducir estas estructuras con fiabilidad.

2. **Arquitectura DCGAN no condicional**: el modelo genera a partir de un vector latente  
   sin ninguna restricción de validez estructural durante el entrenamiento. La función  
   de pérdida adversarial optimiza similitud de distribución global, no reglas locales  
   (p.ej., "pipe TL siempre debe tener pipe TR a su derecha").

3. **Representación one-hot + argmax**: el proceso de selección de tile via `argmax`  
   puede romper transiciones suaves entre tiles de tubería que comparten canales similares.

### Métricas que sí funcionan bien

- **`gap_traversability`** y **`enemy_placement`** son métricas de naturaleza más *local*  
  o *tolerante*: no requieren coherencia de muchos tiles consecutivos.  
- El generador aprendió que los huecos deben ser estrechos (herencia de los niveles  
  de entrenamiento) y que los enemigos aparecen sobre superficies sólidas.

### Conclusión

Este baseline confirma que el DCGAN por sí solo produce niveles visualmente plausibles  
pero estructuralmente incompletos. Las siguientes etapas del proyecto abordarán estas  
limitaciones: **expansión del dataset** (Exp B) y **optimización con CMA-ES** (Exp C).